# Tier 1 prediction analysis

Load one individual-level Tier 1 prediction CSV, inspect descriptive statistics, and compare outcome distributions across demographic groups. The notebook only reads prediction data.

In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 30)
sns.set_theme(style='whitegrid', context='notebook')

OUTCOMES = [
    'trust_multidimensional', 'trust_post', 'distrust_post',
    'funding_perceptions', 'policy_role_mean', 'inst_trust_mean',
    'belief_post', 'concern_mean', 'policy_general',
    'policy_specific_mean', 'behavior_mean', 'donation_ams',
    'newsletter_signup',
]
MODERATORS = {
    'gender': ['Male', 'Female', 'Other'],
    'age_band': ['18-29', '30-44', '45-59', '60+'],
    'race': ['White / Caucasian', 'Black / African American', 'Hispanic / Latino', 'Asian / Asian American', 'Other'],
    'education': ['Less than high school', 'High school diploma / GED', 'Some college or Associate\'s degree', 'Bachelor\'s degree', 'Master\'s degree / Professional degree', 'Doctorate degree / Ph.D.'],
    'income': ['Less than $30,000', '$30,000 to $55,999', '$56,000 to $99,999', '$100,000 to $167,999', '$168,000 or more'],
    'party': ['Republican', 'Democrat', 'Independent', 'Other'],
}

def find_repository_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'predictions').exists():
            return candidate
    raise RuntimeError('Could not find the repository root. Run the notebook from this repository or a child directory.')

REPO_ROOT = find_repository_root()
REPO_ROOT

## Configuration

Change `PREDICTION_FILE` to the Tier 1 CSV you want to analyze. Paths below are repository-relative.

In [ ]:
PREDICTION_FILE = REPO_ROOT / 'predictions' / 'Qwen_Qwen3.5-2B_T1_primary_v1.csv'
PREDICTION_FILE

In [ ]:
def load_t1_predictions(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Tier 1 prediction file not found: {path}')
    if '_T1_' not in path.name:
        raise ValueError(f'Expected a Tier 1 file name containing _T1_, got: {path.name}')

    data = pd.read_csv(path)
    required = {'profile_id', 'condition', *MODERATORS, *OUTCOMES}
    missing = sorted(required.difference(data.columns))
    if missing:
        raise ValueError(f'The Tier 1 file is missing required columns: {missing}')

    for outcome in OUTCOMES:
        numeric = pd.to_numeric(data[outcome], errors='coerce')
        invalid = data[outcome].notna() & numeric.isna()
        if invalid.any():
            raise ValueError(f'{outcome} has {invalid.sum()} non-numeric value(s).')
        data[outcome] = numeric

    condition_order = ['control', *sorted(value for value in data['condition'].dropna().unique() if value != 'control')]
    data['condition'] = pd.Categorical(data['condition'], categories=condition_order, ordered=True)
    for moderator, levels in MODERATORS.items():
        unexpected = sorted(set(data[moderator].dropna()) - set(levels))
        if unexpected:
            raise ValueError(f'{moderator} has unsupported level(s): {unexpected}')
        data[moderator] = pd.Categorical(data[moderator], categories=levels, ordered=True)
    return data, condition_order

df, CONDITION_ORDER = load_t1_predictions(PREDICTION_FILE)
print(f'Loaded {len(df):,} respondents across {df.condition.nunique()} conditions from {PREDICTION_FILE.name}.')
df.head()

In [ ]:
STAT_COLUMNS = ['observations', 'missing', 'mean', 'median', 'variance', 'std_dev', 'minimum', 'maximum']

def descriptive_statistics(data, group_columns):
    group_columns = list(group_columns)
    working = data.copy()
    if not group_columns:
        working['_summary_group'] = 'Overall'
        group_columns = ['_summary_group']
    long_data = working.melt(id_vars=group_columns, value_vars=OUTCOMES, var_name='outcome', value_name='value')
    return (
        long_data.groupby(group_columns + ['outcome'], dropna=False, observed=True)['value']
        .agg(
            observations='count',
            missing=lambda values: int(values.isna().sum()),
            mean='mean',
            median='median',
            variance=lambda values: values.var(ddof=1),
            std_dev=lambda values: values.std(ddof=1),
            minimum='min',
            maximum='max',
        )
        .reset_index()
    )

def show_statistics(table, title):
    print(title)
    display(table.style.format({column: '{:,.3f}' for column in STAT_COLUMNS[2:]}))

overall_statistics = descriptive_statistics(df, [])
condition_statistics = descriptive_statistics(df, ['condition'])
show_statistics(overall_statistics, 'Overall outcome statistics')
show_statistics(condition_statistics, 'Outcome statistics by condition')

In [ ]:
demographic_tables = []
for moderator, levels in MODERATORS.items():
    table = descriptive_statistics(df, ['condition', moderator]).rename(columns={moderator: 'subgroup'})
    table.insert(0, 'demographic', moderator)
    table['_demographic_order'] = list(MODERATORS).index(moderator)
    table['_subgroup_order'] = table['subgroup'].map({level: index for index, level in enumerate(levels)})
    demographic_tables.append(table)

demographic_statistics = pd.concat(demographic_tables, ignore_index=True).sort_values(
    ['_demographic_order', 'condition', '_subgroup_order', 'outcome']
).drop(columns=['_demographic_order', '_subgroup_order'])
show_statistics(demographic_statistics, 'Outcome statistics by condition and demographic subgroup')

In [ ]:
def outcome_label(outcome):
    return outcome.replace('_', ' ')

def plot_distributions_by_demographic(data, moderator, levels):
    plot_data = data.melt(id_vars=[moderator], value_vars=OUTCOMES, var_name='outcome', value_name='value').dropna()
    columns = 3
    rows = math.ceil(len(OUTCOMES) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(18, 4 * rows), squeeze=False)
    for axis, outcome in zip(axes.flat, OUTCOMES):
        selected = plot_data[plot_data['outcome'] == outcome]
        sns.violinplot(
            data=selected, x='value', y=moderator, hue=moderator, order=levels, hue_order=levels,
            density_norm='width', inner='box', cut=0, legend=False, ax=axis,
        )
        axis.set_title(outcome_label(outcome))
        axis.set_xlabel('Predicted value')
        axis.set_ylabel(moderator.replace('_', ' '))
    for axis in axes.flat[len(OUTCOMES):]:
        axis.remove()
    pretty_name = moderator.replace('_', ' ')
    fig.suptitle(f'Tier 1 outcome distributions by {pretty_name}', y=1.01, fontsize=16)
    fig.tight_layout()
    plt.show()

for moderator, levels in MODERATORS.items():
    plot_distributions_by_demographic(df, moderator, levels)